# P1.1 Colab wrapper — official H/I guardrails

**Study:** NANOCHAT-FILIPINO P1.1 · AsPredicted #306780 · ResearchBox #8735  
**Parent RUN_ID:** `p1-20260816T025911Z-0067a57`  
**Split label (every caption):** `reconstructed_article_70_15_15`

This notebook is a **fail-closed host wrapper**. It does not name `gpu_host_for_H_I`. It does not start official Gate H until you type a confirm string after preflight writes `ok: true`. It does not start confirmatory I until official H has passed on **this** named Colab GPU session.

Use it from Cursor with the **Google Colab** extension (kernel → Colab → GPU), or paste it into colab.research.google.com. The plugin only attaches a remote kernel. Local repo files are **not** on `/content` until you upload the frozen handoff zip.

Do **not** put the ResearchBox passcode in this notebook, in Drive, or in a cell output.

## 0. Runtime lock

Runtime → Change runtime type → **GPU**. Not CPU. Not TPU. Then run this cell. If it raises, stop.

In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

def _run(cmd):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        return (p.stdout or p.stderr).strip()
    except (OSError, subprocess.TimeoutExpired) as exc:
        return f"unavailable: {exc}"

if "google.colab" not in sys.modules and not Path("/content").exists():
    print("NOTE: not obviously Colab. Continue only if this process is the intended NVIDIA CUDA host.")

try:
    import torch
except ImportError as exc:
    raise SystemExit(f"torch missing: {exc}") from exc

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() and not torch.cuda.is_available():
    raise SystemExit("MPS is not official H/I. Select a Colab GPU runtime.")
if not torch.cuda.is_available():
    raise SystemExit(
        f"CUDA unavailable. torch={torch.__version__} cuda={torch.version.cuda}. "
        "Runtime must be GPU. CPU/TPU cannot mark Gate H."
    )

RUNTIME = {
    "uname": _run(["uname", "-a"]),
    "machine": platform.machine(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "device": torch.cuda.get_device_name(0),
    "capability": list(torch.cuda.get_device_capability(0)),
    "nvidia_smi": _run(["nvidia-smi"]),
}
print(json.dumps({k: RUNTIME[k] for k in ("machine", "python", "torch", "torch_cuda", "device")}, indent=2))
print(RUNTIME["nvidia_smi"][:1500])
print("\nRe-run this cell after every Colab reconnect. If the GPU name changes, write a host amendment before continuing.")

## 1. Hard bans

These are protocol locks, not style preferences. Later cells read `BANS`.

In [ ]:
BANS = {
    "no_nanochat_dataset": True,          # never: python -m nanochat.dataset
    "no_ratio_minus_1": True,             # never: --target-param-data-ratio=-1
    "no_ratio_12": True,                  # confirmatory I uses per-depth R_d, not 12
    "no_shrink_T": True,                  # T=2048 frozen; shrink --device-batch-size only
    "no_omit_num_iterations_294": True,   # Gate I must pass --num-iterations 294
    "no_test_bpb_before_Dstar": True,
    "no_mark_h_from_cpu_or_mps": True,
    "no_drop_d20_for_oom": True,          # OOM at T=2048 blocks the host, does not drop the depth
    "no_researchbox_passcode": True,
    "no_midrun_resume_as_confirmatory": True,  # Colab disconnect = abort that depth
    "no_tpu": True,
}
FORBIDDEN_SUBSTRINGS = (
    "python -m nanochat.dataset",
    "--target-param-data-ratio=-1",
    "--target-param-data-ratio=12",
    "--max-seq-len=1024",
    "--max-seq-len=512",  # allowed only for official H smoke (d4), not Gate I
)
print("Bans loaded. Do not add ClimbMix, CORE bundle, or test-split training cells below.")

## 2. Isolated paths

Colab `/content` is wiped on disconnect. Checkpoints you care about must be copied to Drive **after** a finished official run. Do not point `NANOCHAT_DATA_DIR` at Drive if the test file lives in the same folder.

In [ ]:
RUN_ID = "p1-20260816T025911Z-0067a57"
NANOCHAT_COMMIT = "92d63d4e8bb4df75c3b71618f31ddde2378b2bcd"
P1_ROOT = Path(os.environ.get("P1_ROOT", "/content/p1/nanochat-filipino")).resolve()

os.environ["P1_ROOT"] = str(P1_ROOT)
os.environ["RUN_ID"] = RUN_ID
os.environ["NANOCHAT_BASE_DIR"] = str(P1_ROOT / "data" / "cache" / RUN_ID)
os.environ["NANOCHAT_DATA_DIR"] = str(P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active")
os.environ["WANDB_RUN"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "1"

data_dir = os.environ["NANOCHAT_DATA_DIR"]
if not data_dir.endswith("wikitext-tl39/active"):
    raise SystemExit(f"NANOCHAT_DATA_DIR must end in wikitext-tl39/active, got {data_dir}")

print("P1_ROOT", P1_ROOT)
print("NANOCHAT_BASE_DIR", os.environ["NANOCHAT_BASE_DIR"])
print("NANOCHAT_DATA_DIR", data_dir)
print("Do not export NANOCHAT_DATA_DIR to the test/ folder.")

## 3. Unpack the frozen handoff (do not rebuild A–G)

Upload `p1.1-dgx-spark-handoff-20260816.zip` to Drive or `/content`, then set `HANDOFF_ZIP` and run. Do not re-download the parquet. Do not re-split. Do not retrain the tokenizer.

In [ ]:
# Optional Drive mount. Keep the test file out of NANOCHAT_DATA_DIR.
MOUNT_DRIVE = False  # set True only if the zip is on Drive
HANDOFF_ZIP = Path("/content/p1.1-dgx-spark-handoff-20260816.zip")

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    # Example: HANDOFF_ZIP = Path("/content/drive/MyDrive/p1.1-dgx-spark-handoff-20260816.zip")

if not P1_ROOT.exists():
    if not HANDOFF_ZIP.is_file():
        raise SystemExit(
            f"Missing {HANDOFF_ZIP}. Upload the frozen zip. "
            "The Cursor Colab plugin does not copy the Mac repo onto /content."
        )
    dest = Path("/content/p1")
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["unzip", "-q", "-o", str(HANDOFF_ZIP), "-d", str(dest)])
    # Zip layout: p1.1-dgx-spark-handoff/nanochat-filipino/...
    found = list(dest.glob("**/nanochat-filipino/manifests/gate_ledger.json"))
    if not found:
        raise SystemExit("Unzip succeeded but nanochat-filipino root was not found.")
    extracted = found[0].parents[1]
    if extracted.resolve() != P1_ROOT:
        P1_ROOT.parent.mkdir(parents=True, exist_ok=True)
        if P1_ROOT.exists():
            raise SystemExit(f"{P1_ROOT} already exists; refuse to overwrite")
        shutil.move(str(extracted), str(P1_ROOT))
    os.environ["P1_ROOT"] = str(P1_ROOT)
    os.environ["NANOCHAT_BASE_DIR"] = str(P1_ROOT / "data" / "cache" / RUN_ID)
    os.environ["NANOCHAT_DATA_DIR"] = str(P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active")

test_jsonl = P1_ROOT / "data" / "processed" / "wikitext-tl39" / "test" / "test.jsonl"
active = P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active"
if not test_jsonl.is_file():
    raise SystemExit(f"isolated test missing: {test_jsonl}")
os.chmod(test_jsonl, 0o444)
test_hits = [p.name for p in active.iterdir() if "test" in p.name.lower()]
if test_hits:
    raise SystemExit(f"test artifacts in active/: {test_hits}")
print("Unpacked at", P1_ROOT)
print("active shards", sorted(p.name for p in active.glob("*.parquet")))
print("test isolated and mode", oct(test_jsonl.stat().st_mode & 0o777))

## 4. Hash lock

Recheck only. Any mismatch = stop. Do not “fix” by re-downloading WikiText-TL-39.

In [ ]:
import hashlib

EXPECTED = {
    "train.parquet": "706d706496e3a085cf4506f97aa8b03faa20d4773d69453eaab4e3ca8f33caf9",
    "train.jsonl": "2b0474c5700dc1eba14def572aa23cc227e4c59c10c2de3ce6b7bda75d137687",
    "val.jsonl": "4d51644b84d05050bfc8c515079e60f6e437082b6cce2122e9ed00e7b1db2b1c",
    "test.jsonl": "3bd193458f4c494d84dae345548c0c01cb6cd7275e98d6ed39a41d517a093baf",
    "shard_00000.parquet": "aaf81d95e577742dcd33a44be2f144c253a5d5650e34b3e622e8b262ff2b6dc9",
    "shard_00001.parquet": "c57c11a2625c38f7f12d1e4018e71bf1f38a56d68fcc9b4952e1b8bded854976",
    "shard_00002.parquet": "13409b3cb78dca87abf1cb1766cd68082b53b704951c38b5d618e97ba7bcfe02",
    "tokenizer.pkl": "04436b854e0841025a3dd2b46baaeeea07a7ccc252e9f99a19171306f00bc5a8",
    "token_bytes.pt": "a5dbc1c88f6292696108263072d77115718cc2d8357f7ad4859adfa517cc2132",
}
PATHS = {
    "train.parquet": P1_ROOT / "data" / "raw" / "wikitext-tl39" / "train.parquet",
    "train.jsonl": P1_ROOT / "data" / "interim" / "wikitext-tl39" / "splits" / "train.jsonl",
    "val.jsonl": P1_ROOT / "data" / "interim" / "wikitext-tl39" / "splits" / "val.jsonl",
    "test.jsonl": P1_ROOT / "data" / "processed" / "wikitext-tl39" / "test" / "test.jsonl",
    "shard_00000.parquet": P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active" / "shard_00000.parquet",
    "shard_00001.parquet": P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active" / "shard_00001.parquet",
    "shard_00002.parquet": P1_ROOT / "data" / "processed" / "wikitext-tl39" / "active" / "shard_00002.parquet",
    "tokenizer.pkl": P1_ROOT / "data" / "cache" / RUN_ID / "tokenizer" / "tokenizer.pkl",
    "token_bytes.pt": P1_ROOT / "data" / "cache" / RUN_ID / "tokenizer" / "token_bytes.pt",
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

HASHES = {}
failed = []
for key, path in PATHS.items():
    if not path.is_file():
        failed.append((key, f"missing {path}"))
        continue
    actual = sha256_file(path)
    HASHES[key] = actual
    ok = actual == EXPECTED[key]
    print(f"{'OK' if ok else 'FAIL'} {key} {actual[:12]}…")
    if not ok:
        failed.append((key, {"expected": EXPECTED[key], "actual": actual}))
if failed:
    raise SystemExit(f"hash lock failed: {failed}")
print("All frozen hashes match.")

## 5. Vendor pin + DATA_DIR hook only

Clone the locked nanochat commit and apply `patches/nanochat-NANOCHAT_DATA_DIR.patch`. Do not edit model, dataloader, tokenizer, split, or evaluator code to “make Colab work.”

In [ ]:
VENDOR = P1_ROOT / "vendor" / "nanochat"
PATCH = P1_ROOT / "patches" / "nanochat-NANOCHAT_DATA_DIR.patch"
VENDOR.parent.mkdir(parents=True, exist_ok=True)

if not (VENDOR / ".git").is_dir():
    subprocess.check_call(["git", "clone", "https://github.com/karpathy/nanochat.git", str(VENDOR)])
subprocess.check_call(["git", "-C", str(VENDOR), "fetch", "--all", "--tags"])
subprocess.check_call(["git", "-C", str(VENDOR), "checkout", NANOCHAT_COMMIT])

dataset_py = (VENDOR / "nanochat" / "dataset.py").read_text(encoding="utf-8")
if "NANOCHAT_DATA_DIR" not in dataset_py:
    subprocess.check_call(["git", "-C", str(VENDOR), "apply", str(PATCH)])
dataset_py = (VENDOR / "nanochat" / "dataset.py").read_text(encoding="utf-8")
if "NANOCHAT_DATA_DIR" not in dataset_py:
    raise SystemExit("DATA_DIR hook missing after patch")

head = subprocess.check_output(["git", "-C", str(VENDOR), "rev-parse", "HEAD"], text=True).strip()
if head != NANOCHAT_COMMIT:
    raise SystemExit(f"wrong nanochat pin: {head}")
print("nanochat", head)
print("hook present")

## 6. CUDA torch pin

Colab’s preinstalled torch may be fine if `torch.cuda.is_available()` is true and you record the exact version. If you reinstall, use an official CUDA wheel. If the install resolves a CPU wheel, **blocked** — do not start H.

In [ ]:
import math
import torch

if not torch.cuda.is_available():
    raise SystemExit("CUDA lost after vendor setup. Classify Colab blocked. Do not start H.")

a = torch.randn(64, 64, device="cuda")
b = torch.randn(64, 64, device="cuda")
c = (a @ b).sum().item()
if not math.isfinite(c):
    raise SystemExit(f"CUDA matmul not finite: {c}")

sys.path.insert(0, str(VENDOR))
from nanochat.gpt import GPT, GPTConfig

inst = []
for depth in (4, 8, 12, 16, 20):
    base_dim = depth * 64
    model_dim = ((base_dim + 127) // 128) * 128
    heads = model_dim // 128
    try:
        with torch.device("cuda"):
            model = GPT(
                GPTConfig(
                    sequence_len=2048,
                    vocab_size=32768,
                    n_layer=depth,
                    n_head=heads,
                    n_kv_head=heads,
                    n_embd=model_dim,
                    window_pattern="SSSL",
                )
            )
            p_total = int(model.num_scaling_params()["total"])
        inst.append({"depth": depth, "ok": True, "p_total": p_total, "t": 2048})
        del model
        torch.cuda.empty_cache()
    except Exception as exc:
        inst.append({"depth": depth, "ok": False, "error": str(exc), "t_shrunk": False})
        # Do not retry at T<2048. Host is blocked for that depth.

print(json.dumps(inst, indent=2))
if not all(row["ok"] for row in inst):
    raise SystemExit(
        "Instantiate failed at T=2048. Classify this Colab GPU blocked for the failed depth. "
        "Do not shrink T. Do not drop d20 from the registered set."
    )
print("All registered depths plus d4 instantiate at T=2048 on this GPU.")

## 7. Write preflight. Do not name the host in this cell.

`--require-pre-i` must still **fail** until official H passes. That is correct.

In [ ]:
from datetime import datetime, timezone

payload = {
    "study_id": "NANOCHAT-FILIPINO-P1.1",
    "purpose": "colab_cuda_preflight_before_naming_host",
    "names_the_official_host": False,
    "official_gate_h_started": False,
    "checked_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "platform": {
        "system": platform.system(),
        "release": platform.release(),
        "machine": platform.machine(),
        "python": platform.python_version(),
    },
    "runtime": RUNTIME,
    "hashes": HASHES,
    "instantiation": inst,
    "bans": BANS,
    "ok": True,
    "next_if_ok": (
        "Human: fill a dated Colab host amendment with this nvidia-smi GPU name, "
        "then set gpu_host_for_H_I. Only then type CONFIRM_OFFICIAL_H in the next cell."
    ),
}
out = P1_ROOT / "manifests" / "colab_host_preflight.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
print("wrote", out)
print("gpu_host_for_H_I stays null until you amend execution_host.json on the Mac copy of the repo.")

# Apparatus check is allowed. --require-pre-i must fail until H.
preflight = P1_ROOT / "scripts" / "p1" / "preflight.py"
if preflight.is_file():
    r = subprocess.run([sys.executable, str(preflight), "--check-apparatus"], cwd=str(P1_ROOT))
    print("--check-apparatus exit", r.returncode)
    r2 = subprocess.run([sys.executable, str(preflight), "--require-pre-i"], cwd=str(P1_ROOT))
    print("--require-pre-i exit", r2.returncode, "(nonzero is correct until official H passes)")
    if r2.returncode == 0:
        raise SystemExit("--require-pre-i passed before official H; do not trust this ledger")

## 8. Official Gate H — gated

Only after:

1. `manifests/colab_host_preflight.json` has `ok: true`
2. You recorded this session’s `nvidia-smi` GPU name in a dated host amendment
3. `gpu_host_for_H_I` names this Colab GPU class

Then set `CONFIRM_OFFICIAL_H = "CONFIRM_OFFICIAL_H"` and run. This is `p1-smoke-d4` only. Fresh-process `base_eval` is the next cell. Never test.

In [ ]:
CONFIRM_OFFICIAL_H = ""  # type CONFIRM_OFFICIAL_H after the host is named

if CONFIRM_OFFICIAL_H != "CONFIRM_OFFICIAL_H":
    raise SystemExit("H is gated. Name the host first. Do not run p1-smoke-d4 as a casual Colab demo.")

pre = json.loads((P1_ROOT / "manifests" / "colab_host_preflight.json").read_text())
if not pre.get("ok"):
    raise SystemExit("preflight not ok")

cmd = [
    sys.executable, "-m", "scripts.base_train",
    "--device-type=cuda",
    "--depth=4",
    "--max-seq-len=512",
    "--device-batch-size=1",
    "--total-batch-size=2048",
    "--num-iterations=30",
    "--eval-tokens=4096",
    "--eval-every=10",
    "--core-metric-every=-1",
    "--sample-every=15",
    "--save-every=30",
    "--model-tag=p1-smoke-d4",
    "--run=dummy",
]
print("cwd", VENDOR)
print(" ".join(cmd))
subprocess.check_call(cmd, cwd=str(VENDOR), env=os.environ.copy())

## 9. Fresh-process H eval (train/val BPB only)

H acceptance still includes H1–H8 from the Spark handoff: finite loss that moved, checkpoint exists, Tagalog-or-garbage sample (not ClimbMix English), no CORE download, CUDA not MPS, test absent from `active/`.

In [ ]:
if CONFIRM_OFFICIAL_H != "CONFIRM_OFFICIAL_H":
    raise SystemExit("H eval is gated on the same confirm string.")

cmd = [
    sys.executable, "-m", "scripts.base_eval",
    "--eval=bpb",
    "--model-tag=p1-smoke-d4",
    "--device-type=cuda",
    "--device-batch-size=1",
    "--split-tokens=4096",
]
subprocess.check_call(cmd, cwd=str(VENDOR), env=os.environ.copy())
print("If the sample talks about the United States Congress, NANOCHAT_DATA_DIR is wrong. Stop.")

## 10. Official Gate I — one card at a time, after H

Use the generated cards in `docs/run-cards/gate-i/`. Shrink `--device-batch-size` only. Keep `--total-batch-size=65536` and `--num-iterations=294`. Never pass ratio `-1`. Disconnect = abort that depth and restart it; do not resume mid-run as confirmatory.

Set `DEPTH` to one of `8, 12, 16, 20` and `CONFIRM_OFFICIAL_I` only after `--require-pre-i` exits 0 on this named host.

In [ ]:
CONFIRM_OFFICIAL_I = ""  # type CONFIRM_OFFICIAL_I after official H pass + --require-pre-i exit 0
DEPTH = None             # 8, 12, 16, or 20
DEVICE_BATCH = 8         # halve until VRAM fits; never change T or B

R_D = {  # from docs/run-cards/gate-i/p1-fixed-d*-3x.md; do not invent
    8: "0.45783403148331536",
    12: "0.17441307843117593",
    16: "0.08175618397870534",
    20: "0.044128661662655576",
}

if CONFIRM_OFFICIAL_I != "CONFIRM_OFFICIAL_I":
    raise SystemExit("I is gated. Official H must pass first.")
if DEPTH not in (8, 12, 16, 20):
    raise SystemExit("DEPTH must be one registered confirmatory depth")
if R_D[DEPTH] in {None, "-1", "12", -1, 12}:
    raise SystemExit("Copy R_d from the generated card. Never -1 or 12.")

cmd = [
    sys.executable, "-m", "scripts.base_train",
    "--device-type=cuda",
    f"--depth={DEPTH}",
    "--max-seq-len=2048",
    f"--device-batch-size={DEVICE_BATCH}",
    "--total-batch-size=65536",
    "--num-iterations=294",
    f"--target-param-data-ratio={R_D[DEPTH]}",
    "--eval-tokens=262144",
    "--eval-every=50",
    "--core-metric-every=-1",
    "--sample-every=200",
    "--save-every=200",
    "--warmup-steps=14",
    f"--run=p1-fixed-d{DEPTH}-3x",
    f"--model-tag=p1-fixed-d{DEPTH}-3x",
]
joined = " ".join(cmd)
if "--target-param-data-ratio=-1" in joined or "nanochat.dataset" in joined:
    raise SystemExit("forbidden token in command")
print(joined)
print("Uncomment the next line only for the named official I run.")
# subprocess.check_call(cmd, cwd=str(VENDOR), env=os.environ.copy())